# Database Architecture Guide for E-commerce Inventory System

## 🎯 Core Requirements Analysis

Based on your business model and data cleaning work, here's why you need **9 core models** (not just 3):

### ✅ **Your Original 3 Models (Enhanced)**
1. **Product** → Master product catalog (MSKU level)
2. **Inventory** → Stock levels per warehouse
3. **Order** → Transactions (inbound/outbound)

### 🆕 **Additional Required Models**
4. **Warehouse** → Fulfillment centers (TLCQ, BLR7, etc.)
5. **Marketplace** → Sales channels (CSTE Amazon, Flipkart, etc.)
6. **SKUMapping** → YES, this needs a model! (Critical for your business)
7. **ComboProduct + ComboProductItem** → YES, combos need models! (Your data has 375 combos)
8. **InventoryMovement** → Audit trail (business requirement)
9. **DataImport** → Track daily report uploads

---

## 🔗 **Why Mapping & Combo SKUs NEED Database Models**

### **SKU Mapping Model is CRITICAL**
```python
# Without DB model: Your current Excel approach
sku_mapper.csv: SKU → MSKU (5,115 mappings)

# With DB model: Proper relational integrity
class SKUMapping(models.Model):
    sku = models.CharField(max_length=100)
    product = models.ForeignKey(Product)  # Enforced MSKU relationship
    marketplace = models.ForeignKey(Marketplace)  # Enforced panel relationship
    # Composite key: (sku, marketplace) - exactly like your cleaned data!
```

**Benefits:**
- ✅ **Referential Integrity**: Can't map to non-existent MSKUs
- ✅ **Performance**: Database indexes for fast lookups
- ✅ **Multi-user Safety**: Concurrent access without file corruption
- ✅ **Audit Trail**: Track who changed what mapping when
- ✅ **Data Validation**: Enforce business rules at database level

### **Combo Products NEED Models**
```python
# Your cleaned combo data: 375 combo products with up to 8 SKUs each
# Without DB model: Complex Excel parsing every time
# With DB model: Clean relational structure

ComboProduct: CSTE_0341_Wands_Hermoine_Sunglasas
├── ComboProductItem 1: CSTE_0291_OT_Hermoine_wand (qty: 1)
└── ComboProductItem 2: CSTE_0319_SG_HarryPotter_Round (qty: 1)
```

**Benefits:**
- ✅ **Dynamic Combos**: Add/remove items without code changes
- ✅ **Stock Calculation**: Auto-calculate combo availability
- ✅ **Order Processing**: Handle combo orders properly
- ✅ **Reporting**: Track combo vs individual sales

---

## 📊 **Database Design Matches Your Cleaned Data Perfectly**

### **Your Cleaned SKU Mappings → SKUMapping Model**
```sql
-- Your Excel: sku, msku, panels, Status 1, Status 2, image
-- Django Model: 
CREATE TABLE sku_mappings (
    sku VARCHAR(100),
    product_id VARCHAR(100) REFERENCES products(msku),
    marketplace_id VARCHAR(20) REFERENCES marketplaces(code),
    status VARCHAR(20),
    image_url TEXT,
    UNIQUE(sku, marketplace_id)  -- Your composite key constraint!
);
```

### **Your Cleaned Inventory → Inventory Model**
```sql
-- Your Excel: Product Name, msku, TLCQ, BLR7, BLR8, BOM5, BOM7, CCU1, etc.
-- Django Models:
CREATE TABLE inventory (
    product_id VARCHAR(100) REFERENCES products(msku),
    warehouse_id VARCHAR(10) REFERENCES warehouses(code),
    current_stock INTEGER,
    UNIQUE(product_id, warehouse_id)
);
```

### **Your Cleaned Combos → ComboProduct + ComboProductItem**
```sql
-- Your Excel: Combo, SKU1, SKU2, SKU3, SKU4, Status
-- Django Models:
CREATE TABLE combo_products (
    combo_sku VARCHAR(100) PRIMARY KEY,
    name VARCHAR(255),
    status VARCHAR(20)
);

CREATE TABLE combo_product_items (
    combo_id VARCHAR(100) REFERENCES combo_products(combo_sku),
    product_id VARCHAR(100) REFERENCES products(msku),
    quantity INTEGER DEFAULT 1,
    position INTEGER
);
```

---

## 🚀 **Django Web App Integration**

### **Data Flow for Your Drag & Drop App**
```python
# User uploads daily_report.xlsx
# 1. DataImport record created (tracking)
# 2. File processed by your InputProcessor
# 3. SKU mapping via database lookups (not Excel)
# 4. Orders created with proper relationships
# 5. Inventory automatically updated via signals
# 6. Outbound data generated from database queries
```

### **Sample Django Views**
```python
# views.py
def upload_daily_report(request):
    if request.method == 'POST':
        uploaded_file = request.FILES['report_file']
        marketplace_code = request.POST['marketplace']
        report_type = request.POST['report_type']  # 'outbound' or 'inbound'
        
        # Create import tracking
        data_import = DataImport.objects.create(
            import_type='DAILY_REPORT',
            marketplace_id=marketplace_code,
            filename=uploaded_file.name,
            file_size=uploaded_file.size,
            imported_by=request.user
        )
        
        # Process with your existing SKUMapper logic
        # But using database models instead of CSV files
        mapper = SKUMapper.from_file(uploaded_file)
        mapper.process_sku_mappings()  # Uses database lookups
        
        # Save to database
        save_orders_to_database(mapper.processed_df, data_import)
        
        return JsonResponse({'status': 'success', 'import_id': data_import.id})

def get_outbound_data(request, import_id):
    # Generate outbound data from database
    orders = Order.objects.filter(
        data_import_id=import_id,
        order_type='OUTBOUND'
    ).select_related('marketplace', 'warehouse')
    
    # Your standardized format: [date, panel, sku, msku, quantity, warehouse]
    outbound_data = []
    for order in orders:
        for item in order.items.all():
            outbound_data.append({
                'date': order.order_date.strftime('%Y-%m-%d'),
                'panel': order.marketplace.code,
                'sku': item.sku,
                'msku': item.product.msku,
                'quantity': item.quantity,
                'warehouse': order.warehouse.code
            })
    
    return JsonResponse({'data': outbound_data})
```

---

## 🔧 **Migration Strategy from Your Current System**

### **1. Initial Data Load**
```python
# management/commands/load_initial_data.py
from django.core.management.base import BaseCommand

class Command(BaseCommand):
    def handle(self, *args, **options):
        # Load your cleaned data into Django models
        
        # 1. Load warehouses
        warehouses = ['TLCQ', 'BLR7', 'BLR8', 'BOM5', 'BOM7', 'CCU1', 
                     'CCX1', 'DEL4', 'DEL5', 'DEX3', 'PNQ2', 'PNQ3', 
                     'SDED', 'SDEE', 'XHJ9']
        for wh_code in warehouses:
            Warehouse.objects.get_or_create(
                code=wh_code,
                defaults={'name': f'Warehouse {wh_code}', 'location': 'TBD'}
            )
        
        # 2. Load marketplaces
        marketplaces = ['CSTE_AMAZON', 'CSTE_FK', 'CSTE_MEESHO', 
                       'GL_FK', 'RUDRAV_MEESHO', 'MISC']
        for mp_code in marketplaces:
            Marketplace.objects.get_or_create(
                code=mp_code,
                defaults={'name': mp_code.replace('_', ' ').title()}
            )
        
        # 3. Load products from cleaned inventory
        self.load_products_from_csv('clean_data/cleaned_inventory.csv')
        
        # 4. Load SKU mappings
        self.load_sku_mappings_from_csv('clean_data/sku_mappings_final_clean.csv')
        
        # 5. Load combo products
        self.load_combos_from_csv('clean_data/combo_sku_clean.csv')
        
        # 6. Load initial inventory
        self.load_inventory_from_csv('clean_data/cleaned_inventory.csv')
```

### **2. Updated SKUMapper Integration**
```python
# Modified core/mapper.py to use Django models instead of CSV files
class DjangoSKUMapper(SKUMapper):
    def __init__(self, report_data_df=None):
        # Initialize without CSV file dependencies
        # Load mapping data from Django models instead
        pass
    
    def map_sku(self, sku, marketplace_code):
        """Map SKU using database lookup"""
        try:
            mapping = SKUMapping.objects.select_related('product').get(
                sku=sku, 
                marketplace_id=marketplace_code
            )
            return mapping.product.msku
        except SKUMapping.DoesNotExist:
            return None
    
    def is_combo_sku(self, sku):
        """Check if SKU is combo using database"""
        return ComboProduct.objects.filter(combo_sku=sku).exists()
    
    def process_combo_sku(self, combo_sku):
        """Get combo items from database"""
        try:
            combo = ComboProduct.objects.prefetch_related('items__product').get(
                combo_sku=combo_sku
            )
            return [item.product.msku for item in combo.items.all()]
        except ComboProduct.DoesNotExist:
            return []
```

---

## 📈 **Benefits of This Database Architecture**

### **🔒 Data Integrity**
- Foreign key constraints prevent orphaned records
- Unique constraints enforce business rules
- Automatic validation at database level

### **⚡ Performance**
- Database indexes for fast lookups
- Optimized queries with select_related/prefetch_related
- No file I/O for every mapping operation

### **👥 Multi-User Support**
- Concurrent access without file corruption
- User tracking for audit trail
- Role-based permissions

### **📊 Advanced Analytics**
- Complex queries across related data
- Real-time inventory tracking
- Sales performance analysis

### **🔧 Maintainability**
- Django admin interface for non-tech users
- Model validations prevent data corruption
- Database migrations for schema changes

---

## 🎯 **Recommended Airtable Alternative: Baserow**

Based on your requirements, **Baserow** is ideal because:

### **✅ Why Baserow is Perfect for Your Use Case**
- **Self-hosted**: Full control over your data
- **Django-based**: Integrates well with your Django app
- **API-first**: Can sync with your Django models
- **Relational**: Supports foreign keys and lookups
- **User-friendly**: Non-tech users can edit data easily

### **🔗 Integration Strategy**
```python
# Sync Django models with Baserow via API
class BaserowSync:
    def sync_products_to_baserow(self):
        # Push Product model data to Baserow for editing
        pass
    
    def sync_mappings_to_baserow(self):
        # Push SKUMapping data for non-tech user editing
        pass
    
    def pull_changes_from_baserow(self):
        # Pull updates back to Django models
        pass
```

---

## 🚀 **Next Steps**

1. **✅ Implement Django Models** (provided above)
2. **✅ Create migrations**: `python manage.py makemigrations`
3. **✅ Load your cleaned data** using management commands
4. **✅ Build drag & drop interface** for daily reports
5. **✅ Set up Baserow** for non-tech user editing
6. **✅ Create dashboard views** for analytics

This architecture gives you:
- ✅ **Scalable foundation** for your e-commerce operations
- ✅ **Professional data management** with proper relationships
- ✅ **Easy integration** with Baserow for non-tech users
- ✅ **Future-proof design** that can grow with your business

Your cleaned data work was excellent preparation for this database design! 🎉